# Full Runner: Multiseed + Sensitivity + Ablation/FRVCP

This notebook runs all three experiment blocks on the full instance set:

1. Multiseed random replication (wireless placement + solver seed).
2. Omega sensitivity with fixed placement seed.
3. Ablation with FRVCP profiling.

Implementation assumptions in current codebase:

- Tenure adaptation is disabled in `evrp_dwc.tabu_search` (fixed tabu tenure).
- Candidate filtering is forced OFF for `n_customers <= 20`.


In [ ]:
from __future__ import annotations

import contextlib
import copy
import hashlib
import io
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd

from frvcpy import core, translator
from frvcpy import solver as frvcp_solver_module

sys.path.insert(0, str(Path("src").resolve()))
from evrp_dwc.tabu_search import (
    RouteEvaluationCache,
    TabuSearch,
    TabuSearchResult,
    build_initial_solution,
    compute_nearest_neighbors,
    multi_start_tabu_search,
)

# -------------------------
# Global config
# -------------------------
DATA_DIR = Path("data/benchmark_xml")
OUTPUT_ROOT = Path("results_raw")
OUTPUT_MULTI = OUTPUT_ROOT / "multiseed_replication"
OUTPUT_SENS = OUTPUT_ROOT / "omega_sensitivity"
OUTPUT_ABLATION = OUTPUT_ROOT / "ablation_frvcp"
OUTPUT_MULTI.mkdir(parents=True, exist_ok=True)
OUTPUT_SENS.mkdir(parents=True, exist_ok=True)
OUTPUT_ABLATION.mkdir(parents=True, exist_ok=True)

INSTANCE_LIMIT: Optional[int] = None
INSTANCE_FILTER: Optional[Set[str]] = None
INSTANCE_SIZES = {10, 20, 40}

# Wireless settings
COVERAGE_LEVELS = [0.10, 0.20, 0.30]
BASE_OMEGA = 0.9
OMEGA_LEVELS = [0.3, 0.5, 0.7, 0.9]

# Multi-seed and sensitivity controls
MULTISEED_PLACEMENT_SEEDS = [20240528 + i for i in range(5)]
SENSITIVITY_FIXED_SEED = 20240528

# Ablation controls
ABLATION_SEED = 20240528
ABLATION_COVERAGES = [0.30]
ABLATION_OMEGA = 0.9

# Execution controls
RUN_MULTI_SEED = True
RUN_SENSITIVITY = True
RUN_ABLATION = True

DEPOT_CHARGING_ALLOWED = True
INITIAL_SOC_FRACTION = 1.0
MULTI_INSERT_ALLOWED = True
DEPOT_ID = 0

USE_MULTI_START = True
NUM_RUNS = 5
NEIGHBOR_LIST_SIZE = 20
MAX_MOVES_PER_ITERATION = 1200

MIN_ENERGY_ABS = 500.0
MIN_ENERGY_FRACTION = 0.05

QUIET_TRANSLATOR_LOGS = True
QUIET_SOLVER_LOGS = True

print("Setup complete")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")
print(f"Coverage levels: {COVERAGE_LEVELS}")
print(f"Omega levels: {OMEGA_LEVELS}")
print(f"Multiseed placement seeds: {MULTISEED_PLACEMENT_SEEDS}")
print(f"Sensitivity fixed seed: {SENSITIVITY_FIXED_SEED}")
print(f"Ablation seed: {ABLATION_SEED}")
print(f"Use multi-start: {USE_MULTI_START} (runs={NUM_RUNS})")


In [ ]:
def _extract_n_from_name(name: str) -> Optional[int]:
    import re

    # Parse benchmark names like tc0c10s2cf1.xml -> n_customers = 10.
    m = re.search(r"tc\d+c(\d+)", name)
    if m:
        return int(m.group(1))

    # Fallback for alternate naming conventions.
    m = re.search(r"c(\d+)s", name)
    return int(m.group(1)) if m else None


def collect_instances(
    data_dir: Path,
    limit: Optional[int] = None,
    include_names: Optional[Set[str]] = None,
    allowed_sizes: Optional[Set[int]] = None,
) -> List[Path]:
    paths = list(data_dir.glob("*.xml"))
    if include_names:
        paths = [p for p in paths if p.name in include_names]
    if allowed_sizes:
        paths = [p for p in paths if _extract_n_from_name(p.name) in allowed_sizes]

    # Force stable order by size first: c10 -> c20 -> c40, then by filename.
    paths = sorted(paths, key=lambda p: ((_extract_n_from_name(p.name) or 10**9), p.name))

    if limit is not None:
        paths = paths[:limit]
    if not paths:
        raise RuntimeError("No instance found with current filters")
    return paths


def paper_tabu_params(n_customers: int, rng_seed: int = 69) -> Dict[str, Any]:
    # shake_iter/phi kept for API compatibility; tenure adaptation is disabled in implementation.
    return dict(
        base_tabu_tenure=5,
        max_iter=100,
        shake_tenure=20,
        shake_iter=20,
        shake_steps=1,
        opt_iter=0,
        phi=0.9,
        rng_seed=rng_seed,
        use_candidate_lists=True,
        use_extended_operators=True,
    )


def translate_instance(xml_path: Path) -> Dict[str, Any]:
    if QUIET_TRANSLATOR_LOGS:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            return translator.translate(xml_path.as_posix(), depot_charging=DEPOT_CHARGING_ALLOWED)
    return translator.translate(xml_path.as_posix(), depot_charging=DEPOT_CHARGING_ALLOWED)


def extract_customers(instance_dict: Dict[str, Any]) -> List[int]:
    inst = core.FrvcpInstance(copy.deepcopy(instance_dict))
    return [
        node.node_id
        for node in inst.nodes_g
        if node.node_id != DEPOT_ID and node.type == core.NodeType.CUSTOMER
    ]


def _instance_selection_seed(instance_name: str, base_seed: int) -> int:
    digest = hashlib.sha256(f"{instance_name}|{base_seed}".encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2**32)


def _customer_candidate_undirected_pairs(instance_dict: Dict[str, Any]) -> List[Tuple[int, int]]:
    process_times = list(instance_dict.get("process_times") or [])
    if not process_times:
        process_times = [0.0] * len(instance_dict["energy_matrix"])
    css_nodes = {cs["node_id"] for cs in instance_dict.get("css", [])}
    customer_ids = [
        node_id
        for node_id in range(len(process_times))
        if node_id != DEPOT_ID and node_id not in css_nodes
    ]
    customer_ids = sorted(customer_ids)
    return [
        (customer_ids[a], customer_ids[b])
        for a in range(len(customer_ids))
        for b in range(a + 1, len(customer_ids))
    ]


def build_incremental_arc_plan(
    instance_dict: Dict[str, Any],
    instance_name: str,
    coverage_levels: Sequence[float],
    base_seed: int,
) -> Dict[str, Any]:
    undirected_pairs = _customer_candidate_undirected_pairs(instance_dict)
    if not undirected_pairs:
        return {
            "candidate_pairs": [],
            "ordered_pairs": [],
            "coverages": list(coverage_levels),
            "selections": [[] for _ in coverage_levels],
        }

    rng = random.Random(_instance_selection_seed(instance_name, base_seed))
    ordered_pairs = undirected_pairs[:]
    rng.shuffle(ordered_pairs)

    selections: List[List[Tuple[int, int]]] = []
    total_pairs = len(ordered_pairs)
    for cov in coverage_levels:
        if cov <= 0.0:
            selections.append([])
            continue
        pair_count = max(1, int(total_pairs * cov))
        pair_count = min(pair_count, total_pairs)
        selected_pairs = ordered_pairs[:pair_count]
        directed: List[Tuple[int, int]] = []
        for i, j in selected_pairs:
            directed.append((i, j))
            directed.append((j, i))
        selections.append(directed)

    return {
        "candidate_pairs": undirected_pairs,
        "ordered_pairs": ordered_pairs,
        "coverages": list(coverage_levels),
        "selections": selections,
    }


def apply_wireless_overlay(
    instance_dict: Dict[str, Any],
    selected_arcs: Sequence[Tuple[int, int]],
    omega: float,
    min_energy_abs: float = MIN_ENERGY_ABS,
    min_energy_fraction: float = MIN_ENERGY_FRACTION,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    inst = copy.deepcopy(instance_dict)
    energy = np.array(inst["energy_matrix"], dtype=float)

    applied = 0
    total_reduction = 0.0
    for i, j in selected_arcs:
        base_energy = float(energy[i, j])
        if base_energy <= 1e-9:
            continue

        min_allowed = max(min_energy_abs, base_energy * min_energy_fraction)
        target_energy = max(min_allowed, base_energy * (1.0 - omega))
        if target_energy >= base_energy - 1e-6:
            continue

        reduction = base_energy - target_energy
        energy[i, j] = target_energy
        applied += 1
        total_reduction += reduction

    inst["energy_matrix"] = energy.tolist()
    return inst, {
        "selected_arcs": len(selected_arcs),
        "applied_arcs": applied,
        "total_reduction": total_reduction,
    }


def solve_instance(
    instance_dict: Dict[str, Any],
    solver_kwargs: Dict[str, Any],
) -> TabuSearchResult:
    frvcp_instance = core.FrvcpInstance(copy.deepcopy(instance_dict))
    customers = [
        node.node_id
        for node in frvcp_instance.nodes_g
        if node.node_id != DEPOT_ID and node.type == core.NodeType.CUSTOMER
    ]
    if not customers:
        raise RuntimeError("No customer in translated instance")

    effective_kwargs = dict(solver_kwargs)
    # Policy: no candidate filtering for c10/c20
    if len(customers) <= 20:
        effective_kwargs["use_candidate_lists"] = False

    initial_soc = min(frvcp_instance.max_q * INITIAL_SOC_FRACTION, frvcp_instance.max_q)
    cache = RouteEvaluationCache(
        instance_dict,
        frvcp_instance,
        initial_soc,
        allow_multi_insert=MULTI_INSERT_ALLOWED,
    )

    use_candidate_lists = bool(effective_kwargs.get("use_candidate_lists", False))
    neighbor_map = (
        compute_nearest_neighbors(frvcp_instance, customers, NEIGHBOR_LIST_SIZE)
        if use_candidate_lists
        else {}
    )

    if USE_MULTI_START:
        return multi_start_tabu_search(
            customers,
            cache,
            required_customers=set(customers),
            num_runs=NUM_RUNS,
            neighbor_map=neighbor_map,
            max_moves_per_iter=MAX_MOVES_PER_ITERATION,
            **effective_kwargs,
        )

    rng = random.Random(effective_kwargs["rng_seed"])
    initial_routes = build_initial_solution(customers, cache, rng, randomized=True, rcl_size=5)
    search = TabuSearch(
        cache=cache,
        all_customers=set(customers),
        neighbor_map=neighbor_map,
        max_moves_per_iter=MAX_MOVES_PER_ITERATION,
        **effective_kwargs,
    )
    return search.run(initial_routes)


class FRVCPProfiler:
    def __init__(self):
        self.calls = 0
        self.total_time = 0.0
        self._orig = None

    def __enter__(self):
        self._orig = frvcp_solver_module.Solver.solve
        profiler = self

        def wrapped_solve(solver_obj, *args, **kwargs):
            t0 = time.perf_counter()
            try:
                return profiler._orig(solver_obj, *args, **kwargs)
            finally:
                profiler.calls += 1
                profiler.total_time += (time.perf_counter() - t0)

        frvcp_solver_module.Solver.solve = wrapped_solve
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        frvcp_solver_module.Solver.solve = self._orig


def solve_overlay_case(
    base_instance: Dict[str, Any],
    instance_name: str,
    n_customers: int,
    coverage: float,
    omega: float,
    selected_arcs: Sequence[Tuple[int, int]],
    candidate_pair_count: int,
    placement_seed: int,
    solver_seed: int,
    solver_overrides: Optional[Dict[str, Any]] = None,
    profile_frvcp: bool = False,
) -> Dict[str, Any]:
    if coverage <= 0.0 or omega is None or len(selected_arcs) == 0:
        test_instance = copy.deepcopy(base_instance)
        overlay = {"selected_arcs": 0, "applied_arcs": 0, "total_reduction": 0.0}
    else:
        test_instance, overlay = apply_wireless_overlay(base_instance, selected_arcs, omega)

    solver_kwargs = paper_tabu_params(n_customers=n_customers, rng_seed=solver_seed)
    if solver_overrides:
        solver_kwargs.update(solver_overrides)

    # Policy repeated in output fields for traceability
    effective_use_candidate_lists = bool(solver_kwargs.get("use_candidate_lists", False)) and n_customers > 20

    t0 = time.time()
    result = None
    err = None
    frvcp_calls = None
    frvcp_total_time = None

    try:
        if profile_frvcp:
            with FRVCPProfiler() as profiler:
                if QUIET_SOLVER_LOGS:
                    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                        result = solve_instance(test_instance, solver_kwargs)
                else:
                    result = solve_instance(test_instance, solver_kwargs)
            frvcp_calls = profiler.calls
            frvcp_total_time = profiler.total_time
        else:
            if QUIET_SOLVER_LOGS:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    result = solve_instance(test_instance, solver_kwargs)
            else:
                result = solve_instance(test_instance, solver_kwargs)
    except Exception as e:
        err = str(e)

    elapsed = time.time() - t0

    return {
        "instance": instance_name,
        "n_customers": n_customers,
        "coverage": coverage,
        "coverage_pct": f"{int(round(coverage * 100))}%",
        "omega": omega,
        "placement_seed": placement_seed,
        "solver_seed": solver_seed,
        "base_tabu_tenure": solver_kwargs["base_tabu_tenure"],
        "max_iter": solver_kwargs["max_iter"],
        "shake_tenure": solver_kwargs["shake_tenure"],
        "shake_iter": solver_kwargs["shake_iter"],
        "shake_steps": solver_kwargs.get("shake_steps", 1),
        "opt_iter": solver_kwargs["opt_iter"],
        "phi": solver_kwargs["phi"],
        "use_candidate_lists": effective_use_candidate_lists,
        "use_extended_operators": solver_kwargs.get("use_extended_operators", False),
        "best_cost": None if result is None else result.best_cost,
        "vehicles": None if result is None else len(result.routes),
        "iterations_done": 0 if result is None else len(result.history),
        "moves_generated": None if result is None else result.stats.get("moves_generated"),
        "moves_evaluated": None if result is None else result.stats.get("moves_evaluated"),
        "moves_skipped_validity": None if result is None else result.stats.get("moves_skipped_validity"),
        "moves_skipped_tabu": None if result is None else result.stats.get("moves_skipped_tabu"),
        "shakes": None if result is None else result.stats.get("shakes"),
        "solver_time_sec": elapsed,
        "candidate_pairs": candidate_pair_count,
        "selected_arcs": overlay["selected_arcs"],
        "applied_arcs": overlay["applied_arcs"],
        "total_reduction": overlay["total_reduction"],
        "frvcp_calls": frvcp_calls,
        "frvcp_total_time_sec": frvcp_total_time,
        "frvcp_avg_time_ms": None if not frvcp_calls else (frvcp_total_time / frvcp_calls) * 1000.0,
        "error": err,
    }


def save_seed_results(
    rows: List[Dict[str, Any]],
    seed_file: Path,
    aggregate_file: Path,
    dedup_keys: Sequence[str],
) -> pd.DataFrame:
    seed_df = pd.DataFrame(rows)
    seed_df.to_csv(seed_file, index=False)

    if aggregate_file.exists():
        agg_df = pd.read_csv(aggregate_file)
        combined = pd.concat([agg_df, seed_df], ignore_index=True)
    else:
        combined = seed_df.copy()

    combined = combined.drop_duplicates(subset=list(dedup_keys), keep="last")
    combined.to_csv(aggregate_file, index=False)
    return combined


print("Helpers ready")


In [ ]:
INSTANCE_PATHS = collect_instances(
    DATA_DIR,
    limit=INSTANCE_LIMIT,
    include_names=INSTANCE_FILTER,
    allowed_sizes=INSTANCE_SIZES,
)

print(f"Selected {len(INSTANCE_PATHS)} instances")
for p in INSTANCE_PATHS[:10]:
    print(" -", p.name)
if len(INSTANCE_PATHS) > 10:
    print(" ...")


In [ ]:
# -------------------------
# Experiment 0: No-wireless baseline (all instances)
# -------------------------
RUN_NO_WIRELESS_BASELINE = True
NO_WIRELESS_SOLVER_SEED = 20240528
PROFILE_FRVCP_NO_WIRELESS = False

OUTPUT_BASELINE = OUTPUT_ROOT / "no_wireless_baseline"
OUTPUT_BASELINE.mkdir(parents=True, exist_ok=True)

if RUN_NO_WIRELESS_BASELINE:
    out_file = OUTPUT_BASELINE / "no_wireless_all.csv"
    rows: List[Dict[str, Any]] = []

    print(f"[NO-WIRELESS] instances={len(INSTANCE_PATHS)} | seed={NO_WIRELESS_SOLVER_SEED}")
    for idx, instance_path in enumerate(INSTANCE_PATHS, 1):
        t_inst = time.time()
        try:
            base_instance = translate_instance(instance_path)
            customers = extract_customers(base_instance)
            n_customers = len(customers)

            row = solve_overlay_case(
                base_instance=base_instance,
                instance_name=instance_path.name,
                n_customers=n_customers,
                coverage=0.0,
                omega=0.0,
                selected_arcs=[],
                candidate_pair_count=0,
                placement_seed=NO_WIRELESS_SOLVER_SEED,
                solver_seed=NO_WIRELESS_SOLVER_SEED,
                solver_overrides=None,
                profile_frvcp=PROFILE_FRVCP_NO_WIRELESS,
            )
            row["experiment"] = "no_wireless_baseline"
            rows.append(row)

        except Exception as e:
            rows.append(
                {
                    "instance": instance_path.name,
                    "n_customers": _extract_n_from_name(instance_path.name),
                    "coverage": 0.0,
                    "coverage_pct": "0%",
                    "omega": 0.0,
                    "placement_seed": NO_WIRELESS_SOLVER_SEED,
                    "solver_seed": NO_WIRELESS_SOLVER_SEED,
                    "experiment": "no_wireless_baseline",
                    "error": f"load_or_run_error: {e}",
                }
            )

        print(f"  [{idx}/{len(INSTANCE_PATHS)}] {instance_path.name} ({time.time() - t_inst:.1f}s)")

    no_wireless_df = pd.DataFrame(rows)
    no_wireless_df.to_csv(out_file, index=False)

    print(f"[SAVED] {out_file} | rows={len(no_wireless_df)}")
    ok = no_wireless_df[no_wireless_df["error"].isna()] if "error" in no_wireless_df.columns else no_wireless_df
    if len(ok) > 0:
        display(
            ok.groupby("n_customers", as_index=False).agg(
                n=("instance", "count"),
                mean_cost=("best_cost", "mean"),
                mean_vehicles=("vehicles", "mean"),
                mean_time_sec=("solver_time_sec", "mean"),
            )
        )
else:
    print("RUN_NO_WIRELESS_BASELINE=False -> skipped")


In [ ]:
# -------------------------
# Experiment 1: Multiseed random replication
# -------------------------
if RUN_MULTI_SEED:
    aggregate_file = OUTPUT_MULTI / "multiseed_all.csv"
    dedup_keys = ["instance", "coverage", "placement_seed", "solver_seed", "omega"]

    for seed in MULTISEED_PLACEMENT_SEEDS:
        seed_file = OUTPUT_MULTI / f"multiseed_seed_{seed}.csv"
        if seed_file.exists():
            print(f"[SKIP] {seed_file.name} already exists")
            continue

        print(f"[RUN] multiseed seed={seed}, omega={BASE_OMEGA}")
        rows: List[Dict[str, Any]] = []

        for idx, instance_path in enumerate(INSTANCE_PATHS, 1):
            t_inst = time.time()
            try:
                base_instance = translate_instance(instance_path)
                customers = extract_customers(base_instance)
                n_customers = len(customers)
                plan = build_incremental_arc_plan(
                    base_instance,
                    instance_path.name,
                    COVERAGE_LEVELS,
                    base_seed=seed,
                )
                candidate_pair_count = len(plan["candidate_pairs"])

                for coverage, selected_arcs in zip(COVERAGE_LEVELS, plan["selections"]):
                    row = solve_overlay_case(
                        base_instance=base_instance,
                        instance_name=instance_path.name,
                        n_customers=n_customers,
                        coverage=coverage,
                        omega=BASE_OMEGA,
                        selected_arcs=selected_arcs,
                        candidate_pair_count=candidate_pair_count,
                        placement_seed=seed,
                        solver_seed=seed,
                        solver_overrides=None,
                        profile_frvcp=False,
                    )
                    row["experiment"] = "multiseed_replication"
                    rows.append(row)
            except Exception as e:
                print(f"[ERROR] {instance_path.name}: {e}")

            print(f"  [{idx}/{len(INSTANCE_PATHS)}] {instance_path.name} ({time.time() - t_inst:.1f}s)")

        combined_df = save_seed_results(rows, seed_file, aggregate_file, dedup_keys=dedup_keys)
        print(f"[SAVED] {seed_file}")
        print(f"[UPDATED] {aggregate_file} | rows={len(combined_df)}")

    if aggregate_file.exists():
        df = pd.read_csv(aggregate_file)
        print("\nMultiseed quick summary:")
        display(df.groupby(["coverage", "placement_seed"], as_index=False)["best_cost"].mean().head(12))
else:
    print("RUN_MULTI_SEED=False -> skipped")


In [ ]:
# -------------------------
# Experiment 2: Omega sensitivity (fixed seed)
# -------------------------
if RUN_SENSITIVITY:
    aggregate_file = OUTPUT_SENS / "omega_sensitivity_all.csv"
    dedup_keys = ["instance", "coverage", "placement_seed", "solver_seed", "omega"]

    seed = SENSITIVITY_FIXED_SEED
    for omega in OMEGA_LEVELS:
        seed_file = OUTPUT_SENS / f"omega_{omega:.1f}_seed_{seed}.csv"
        if seed_file.exists():
            print(f"[SKIP] {seed_file.name} already exists")
            continue

        print(f"[RUN] sensitivity omega={omega:.1f}, fixed seed={seed}")
        rows: List[Dict[str, Any]] = []

        for idx, instance_path in enumerate(INSTANCE_PATHS, 1):
            t_inst = time.time()
            try:
                base_instance = translate_instance(instance_path)
                customers = extract_customers(base_instance)
                n_customers = len(customers)
                plan = build_incremental_arc_plan(
                    base_instance,
                    instance_path.name,
                    COVERAGE_LEVELS,
                    base_seed=seed,
                )
                candidate_pair_count = len(plan["candidate_pairs"])

                for coverage, selected_arcs in zip(COVERAGE_LEVELS, plan["selections"]):
                    row = solve_overlay_case(
                        base_instance=base_instance,
                        instance_name=instance_path.name,
                        n_customers=n_customers,
                        coverage=coverage,
                        omega=omega,
                        selected_arcs=selected_arcs,
                        candidate_pair_count=candidate_pair_count,
                        placement_seed=seed,
                        solver_seed=seed,
                        solver_overrides=None,
                        profile_frvcp=False,
                    )
                    row["experiment"] = "omega_sensitivity"
                    rows.append(row)
            except Exception as e:
                print(f"[ERROR] {instance_path.name}: {e}")

            print(f"  [{idx}/{len(INSTANCE_PATHS)}] {instance_path.name} ({time.time() - t_inst:.1f}s)")

        combined_df = save_seed_results(rows, seed_file, aggregate_file, dedup_keys=dedup_keys)
        print(f"[SAVED] {seed_file}")
        print(f"[UPDATED] {aggregate_file} | rows={len(combined_df)}")

    if aggregate_file.exists():
        df = pd.read_csv(aggregate_file)
        print("\nSensitivity quick summary:")
        display(df.groupby(["omega", "coverage"], as_index=False)["best_cost"].mean())
else:
    print("RUN_SENSITIVITY=False -> skipped")


In [ ]:
# -------------------------
# Experiment 3: Ablation + FRVCP profiling
# -------------------------
ABLATION_CONFIGS = [
    {
        "ablation": "baseline",
        "overrides": {},
    },
    {
        "ablation": "no_shaking",
        "overrides": {"shake_tenure": 10**9},
    },
    {
        "ablation": "no_candidate_filtering",
        "overrides": {"use_candidate_lists": False},
    },
    {
        "ablation": "no_extended_operators",
        "overrides": {"use_extended_operators": False},
    },
    {
        "ablation": "no_filter_no_extended",
        "overrides": {"use_candidate_lists": False, "use_extended_operators": False},
    },
]

if RUN_ABLATION:
    raw_file = OUTPUT_ABLATION / "ablation_frvcp_raw.csv"
    summary_file = OUTPUT_ABLATION / "ablation_frvcp_summary.csv"

    rows: List[Dict[str, Any]] = []
    print(f"[ABLATION] instances={len(INSTANCE_PATHS)} | coverages={ABLATION_COVERAGES} | omega={ABLATION_OMEGA} | seed={ABLATION_SEED}")

    for idx, instance_path in enumerate(INSTANCE_PATHS, 1):
        t_inst = time.time()
        try:
            base_instance = translate_instance(instance_path)
            customers = extract_customers(base_instance)
            n_customers = len(customers)

            plan = build_incremental_arc_plan(
                base_instance,
                instance_path.name,
                ABLATION_COVERAGES,
                base_seed=ABLATION_SEED,
            )
            candidate_pair_count = len(plan["candidate_pairs"])

            for coverage, selected_arcs in zip(ABLATION_COVERAGES, plan["selections"]):
                for cfg in ABLATION_CONFIGS:
                    # Candidate-filter ablations are redundant on c10/c20 by design.
                    if n_customers <= 20 and cfg["ablation"] in {"no_candidate_filtering", "no_filter_no_extended"}:
                        continue

                    row = solve_overlay_case(
                        base_instance=base_instance,
                        instance_name=instance_path.name,
                        n_customers=n_customers,
                        coverage=coverage,
                        omega=ABLATION_OMEGA,
                        selected_arcs=selected_arcs,
                        candidate_pair_count=candidate_pair_count,
                        placement_seed=ABLATION_SEED,
                        solver_seed=ABLATION_SEED,
                        solver_overrides=cfg["overrides"],
                        profile_frvcp=True,
                    )
                    row["experiment"] = "ablation_frvcp"
                    row["ablation"] = cfg["ablation"]
                    rows.append(row)

        except Exception as e:
            print(f"[ERROR] {instance_path.name}: {e}")

        print(f"  [{idx}/{len(INSTANCE_PATHS)}] {instance_path.name} ({time.time() - t_inst:.1f}s)")

    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(raw_file, index=False)

    ok = raw_df[raw_df["error"].isna()].copy()
    summary_df = (
        ok.groupby("ablation", as_index=False)
        .agg(
            n=("instance", "count"),
            mean_cost=("best_cost", "mean"),
            mean_vehicles=("vehicles", "mean"),
            mean_solver_time_sec=("solver_time_sec", "mean"),
            mean_frvcp_calls=("frvcp_calls", "mean"),
            mean_frvcp_ms=("frvcp_avg_time_ms", "mean"),
        )
        .sort_values("ablation")
    )
    summary_df.to_csv(summary_file, index=False)

    print(f"[SAVED] {raw_file} | rows={len(raw_df)}")
    print(f"[SAVED] {summary_file} | rows={len(summary_df)}")
    display(summary_df)
else:
    print("RUN_ABLATION=False -> skipped")


In [ ]:
# -------------------------
# Optional Experiment 3b: 5-solver-seed ablation with fixed placement (resume mode)
# -------------------------
# This cell can extend the existing single-seed ablation to five solver seeds
# without recomputing the seed that has already been run. It first imports the
# existing seed 20240528 rows from Experiment 3, then runs only missing
# (instance, coverage, solver_seed, ablation) combinations.

RUN_ABLATION_5_SEED = False
ABLATION_5_SEED_SOLVER_SEEDS = [20240528 + i for i in range(5)]
ABLATION_5_SEED_PLACEMENT_SEED = 20240528
OUTPUT_ABLATION_5_SEED = OUTPUT_ROOT / "ablation_frvcp_5_solver_seeds"
OUTPUT_ABLATION_5_SEED.mkdir(parents=True, exist_ok=True)
ABLATION_5_SEED_RAW_FILE = OUTPUT_ABLATION_5_SEED / "ablation_frvcp_5_solver_seeds_raw.csv"
ABLATION_5_SEED_SUMMARY_FILE = OUTPUT_ABLATION_5_SEED / "ablation_frvcp_5_solver_seeds_summary.csv"
ABLATION_5_SEED_PAIRED_FILE = OUTPUT_ABLATION_5_SEED / "ablation_frvcp_5_solver_seeds_paired.csv"

ABLATION_ORIGINAL_RAW_FILE = OUTPUT_ABLATION / "ablation_frvcp_raw.csv"
ABLATION_ORIGINAL_SUMMARY_FILE = OUTPUT_ABLATION / "ablation_frvcp_summary.csv"
ABLATION_SINGLE_SEED_REFERENCE = OUTPUT_ROOT / "ablation_frvcp_single_seed_reference"
ABLATION_SINGLE_SEED_SUMMARY_FILE = ABLATION_SINGLE_SEED_REFERENCE / "ablation_frvcp_single_seed_summary.csv"
ABLATION_SINGLE_SEED_PAIRED_FILE = ABLATION_SINGLE_SEED_REFERENCE / "ablation_frvcp_single_seed_paired.csv"

# Keep the same variants as the original ablation. Candidate-filter variants are
# still skipped on c10/c20, matching Experiment 3.
ABLATION_5_SEED_CONFIGS = [
    {"ablation": "baseline", "overrides": {}},
    {"ablation": "no_shaking", "overrides": {"shake_tenure": 10**9}},
    {"ablation": "no_candidate_filtering", "overrides": {"use_candidate_lists": False}},
    {"ablation": "no_extended_operators", "overrides": {"use_extended_operators": False}},
    {"ablation": "no_filter_no_extended", "overrides": {"use_candidate_lists": False, "use_extended_operators": False}},
]


def _ablation_key(row: pd.Series) -> tuple:
    return (
        row.get("instance"),
        row.get("coverage_pct"),
        int(row.get("solver_seed")),
        row.get("ablation"),
    )


def _write_ablation_5_seed_outputs(raw_df: pd.DataFrame) -> None:
    raw_df.to_csv(ABLATION_5_SEED_RAW_FILE, index=False)
    ok = raw_df[raw_df["error"].isna()].copy() if "error" in raw_df.columns else raw_df.copy()

    if ok.empty:
        print("No successful rows to summarize yet.")
        return

    summary_df = (
        ok.groupby("ablation", as_index=False)
        .agg(
            n=("instance", "count"),
            mean_cost=("best_cost", "mean"),
            std_cost=("best_cost", "std"),
            mean_vehicles=("vehicles", "mean"),
            std_vehicles=("vehicles", "std"),
            mean_solver_time_sec=("solver_time_sec", "mean"),
            mean_frvcp_calls=("frvcp_calls", "mean"),
            mean_frvcp_ms=("frvcp_avg_time_ms", "mean"),
            mean_shakes=("shakes", "mean"),
        )
        .sort_values("ablation")
    )
    summary_df.to_csv(ABLATION_5_SEED_SUMMARY_FILE, index=False)

    baseline = ok[ok["ablation"] == "baseline"].copy()
    variants = ok[ok["ablation"] != "baseline"].copy()
    if not baseline.empty and not variants.empty:
        paired_df = variants.merge(
            baseline[["instance", "coverage_pct", "solver_seed", "best_cost", "vehicles"]],
            on=["instance", "coverage_pct", "solver_seed"],
            suffixes=("", "_baseline"),
        )
        paired_df["delta_cost_pct_vs_baseline"] = (
            (paired_df["best_cost"] - paired_df["best_cost_baseline"])
            / paired_df["best_cost_baseline"]
            * 100
        )
        paired_df["delta_vehicles_vs_baseline"] = paired_df["vehicles"] - paired_df["vehicles_baseline"]
        paired_df.to_csv(ABLATION_5_SEED_PAIRED_FILE, index=False)


def _display_ablation_outputs(summary_file: Path, paired_file: Optional[Path] = None) -> None:
    display(pd.read_csv(summary_file))
    if paired_file is not None and paired_file.exists():
        paired_df = pd.read_csv(paired_file)
        display(
            paired_df.groupby("ablation", as_index=False)
            .agg(
                n=("instance", "count"),
                mean_delta_cost_pct=("delta_cost_pct_vs_baseline", "mean"),
                std_delta_cost_pct=("delta_cost_pct_vs_baseline", "std"),
                worsened_runs=("delta_cost_pct_vs_baseline", lambda s: int((s > 1e-9).sum())),
                improved_runs=("delta_cost_pct_vs_baseline", lambda s: int((s < -1e-9).sum())),
                mean_delta_vehicles=("delta_vehicles_vs_baseline", "mean"),
            )
        )


if RUN_ABLATION_5_SEED:
    if ABLATION_5_SEED_RAW_FILE.exists():
        raw_df = pd.read_csv(ABLATION_5_SEED_RAW_FILE)
        print(f"[RESUME] Loaded existing {ABLATION_5_SEED_RAW_FILE} | rows={len(raw_df)}")
    else:
        raw_df = pd.DataFrame()

    # Seed the 5-seed raw file with the already completed single-seed ablation.
    if ABLATION_ORIGINAL_RAW_FILE.exists():
        original_df = pd.read_csv(ABLATION_ORIGINAL_RAW_FILE)
        original_seed_df = original_df[
            (original_df["placement_seed"] == ABLATION_5_SEED_PLACEMENT_SEED)
            & (original_df["solver_seed"] == ABLATION_5_SEED_PLACEMENT_SEED)
        ].copy()
        original_seed_df["experiment"] = "ablation_frvcp_5_solver_seeds"
        original_seed_df["seed_source"] = "imported_from_single_seed_ablation"

        existing_keys = set()
        if not raw_df.empty:
            existing_keys = set(raw_df.apply(_ablation_key, axis=1))
        seed_rows = original_seed_df[
            ~original_seed_df.apply(_ablation_key, axis=1).isin(existing_keys)
        ]
        if not seed_rows.empty:
            raw_df = pd.concat([raw_df, seed_rows], ignore_index=True, sort=False)
            _write_ablation_5_seed_outputs(raw_df)
            print(f"[SEEDED] Imported existing solver_seed={ABLATION_5_SEED_PLACEMENT_SEED} rows={len(seed_rows)}")
    else:
        print(f"[WARN] Original single-seed ablation not found: {ABLATION_ORIGINAL_RAW_FILE}")

    done_keys = set(raw_df.apply(_ablation_key, axis=1)) if not raw_df.empty else set()
    print(
        "[ABLATION-5-SEED RESUME] "
        f"instances={len(INSTANCE_PATHS)} | coverages={ABLATION_COVERAGES} | omega={ABLATION_OMEGA} | "
        f"placement_seed={ABLATION_5_SEED_PLACEMENT_SEED} | solver_seeds={ABLATION_5_SEED_SOLVER_SEEDS} | "
        f"existing_rows={len(raw_df)}"
    )

    new_rows: List[Dict[str, Any]] = []
    for idx, instance_path in enumerate(INSTANCE_PATHS, 1):
        t_inst = time.time()
        try:
            base_instance = translate_instance(instance_path)
            customers = extract_customers(base_instance)
            n_customers = len(customers)

            # Fixed DWC placement across solver seeds, so differences isolate solver randomness.
            plan = build_incremental_arc_plan(
                base_instance,
                instance_path.name,
                ABLATION_COVERAGES,
                base_seed=ABLATION_5_SEED_PLACEMENT_SEED,
            )
            candidate_pair_count = len(plan["candidate_pairs"])

            for coverage, selected_arcs in zip(ABLATION_COVERAGES, plan["selections"]):
                coverage_pct = f"{int(round(coverage * 100))}%"
                for solver_seed in ABLATION_5_SEED_SOLVER_SEEDS:
                    for cfg in ABLATION_5_SEED_CONFIGS:
                        if n_customers <= 20 and cfg["ablation"] in {"no_candidate_filtering", "no_filter_no_extended"}:
                            continue

                        key = (instance_path.name, coverage_pct, int(solver_seed), cfg["ablation"])
                        if key in done_keys:
                            continue

                        row = solve_overlay_case(
                            base_instance=base_instance,
                            instance_name=instance_path.name,
                            n_customers=n_customers,
                            coverage=coverage,
                            omega=ABLATION_OMEGA,
                            selected_arcs=selected_arcs,
                            candidate_pair_count=candidate_pair_count,
                            placement_seed=ABLATION_5_SEED_PLACEMENT_SEED,
                            solver_seed=solver_seed,
                            solver_overrides=cfg["overrides"],
                            profile_frvcp=True,
                        )
                        row["experiment"] = "ablation_frvcp_5_solver_seeds"
                        row["ablation"] = cfg["ablation"]
                        row["seed_source"] = "computed"
                        new_rows.append(row)
                        done_keys.add(key)

        except Exception as e:
            print(f"[ERROR] {instance_path.name}: {e}")

        if new_rows:
            raw_df = pd.concat([raw_df, pd.DataFrame(new_rows)], ignore_index=True, sort=False)
            new_rows = []
            _write_ablation_5_seed_outputs(raw_df)

        print(f"  [{idx}/{len(INSTANCE_PATHS)}] {instance_path.name} ({time.time() - t_inst:.1f}s) | rows={len(raw_df)}")

    _write_ablation_5_seed_outputs(raw_df)
    print(f"[SAVED] {ABLATION_5_SEED_RAW_FILE} | rows={len(raw_df)}")
    print(f"[SAVED] {ABLATION_5_SEED_SUMMARY_FILE}")
    print(f"[SAVED] {ABLATION_5_SEED_PAIRED_FILE}")
    _display_ablation_outputs(ABLATION_5_SEED_SUMMARY_FILE, ABLATION_5_SEED_PAIRED_FILE)
else:
    if ABLATION_5_SEED_SUMMARY_FILE.exists():
        print("RUN_ABLATION_5_SEED=False -> loading existing optional 5-solver-seed ablation summary")
        _display_ablation_outputs(ABLATION_5_SEED_SUMMARY_FILE, ABLATION_5_SEED_PAIRED_FILE)
    elif ABLATION_SINGLE_SEED_SUMMARY_FILE.exists():
        print("RUN_ABLATION_5_SEED=False -> no 5-seed output found; loading existing single-seed ablation reference")
        _display_ablation_outputs(ABLATION_SINGLE_SEED_SUMMARY_FILE, ABLATION_SINGLE_SEED_PAIRED_FILE)
    elif ABLATION_ORIGINAL_SUMMARY_FILE.exists():
        print("RUN_ABLATION_5_SEED=False -> no 5-seed/reference output found; loading original single-seed ablation")
        display(pd.read_csv(ABLATION_ORIGINAL_SUMMARY_FILE))
    else:
        print("RUN_ABLATION_5_SEED=False -> optional 5-solver-seed ablation skipped; no existing output found")


In [ ]:
# -------------------------
# Optional: consolidated snapshot
# -------------------------
multi_file = OUTPUT_MULTI / "multiseed_all.csv"
sens_file = OUTPUT_SENS / "omega_sensitivity_all.csv"
abl_file = OUTPUT_ABLATION / "ablation_frvcp_raw.csv"
abl5_file = OUTPUT_ROOT / "ablation_frvcp_5_solver_seeds" / "ablation_frvcp_5_solver_seeds_raw.csv"

frames = []
for f in [multi_file, sens_file, abl_file, abl5_file]:
    if f.exists():
        df = pd.read_csv(f)
        df["source_file"] = f.name
        frames.append(df)

if frames:
    combined = pd.concat(frames, ignore_index=True, sort=False)
    out_file = OUTPUT_ROOT / "all_experiments_snapshot.csv"
    combined.to_csv(out_file, index=False)
    print(f"[SAVED] {out_file} | rows={len(combined)}")
    display(combined.head(10))
else:
    print("No experiment output found yet.")


## Notes

- For quick dry-run, set `INSTANCE_LIMIT` to a small value (for example `3`).
- To reduce runtime, set `USE_MULTI_START=False` and keep `NUM_RUNS=1`.
- You can rerun one experiment cell only; per-seed CSV files are saved for resume.
